# Email copilot: end-to-end demo

A single-user productivity copilot on top of one existing mailbox and calendar, not a mail system.

This notebook runs the end-to-end demo. Human decisions are scripted so the run is
reproducible. With the default `LLM_PROVIDER=fake` it needs no API keys and gives the same output every time.
Nothing is really sent: mail and calendar go through mock adapters, and everything is written to a throwaway folder.

In [1]:
import json
import shutil
import sys
import tempfile
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from assistant.cli import render_view
from assistant.config import Settings
from assistant.deps import build_deps
from assistant.graph import build_graph, emails_to_process, pending_view, process_email
from assistant.schemas import HumanDecision

# Logs, traces and metrics for this run go to logs/demo/ (sample_logs/ in the repo is a copy of one run).
LOGS = ROOT / "logs" / "demo"
shutil.rmtree(LOGS, ignore_errors=True)
RUNTIME = Path(tempfile.mkdtemp(prefix="copilot-demo-"))  # mock sent.json, calendar.json and memory.db

settings = Settings(data_dir=ROOT / "data", runtime_dir=RUNTIME, logs_dir=LOGS)
deps = build_deps(settings)
app = build_graph()
print(f"Provider: {settings.llm_provider}, knowledge search: {settings.knowledge_backend}")

Provider: fake, knowledge search: chroma


## 1. Seeded memory

Long-term memory starts from `data/seed_memory.json`: preferences and contacts (Sofia Lind is a VIP).
Organisational facts come from `data/org_facts.md` and are searched through Chroma.

In [2]:
for p in deps.memory.preference_records():
    print(f"preference  {p.key:24} {p.value!r:28} ({p.source})")
for email in ["sofia.lind@example.com", "mark.jensen@nordwind.example"]:
    c = deps.memory.get_contact(email)
    print(f"contact     {c.email:32} {c.relationship}{'  VIP' if c.vip else ''}")
print("org facts  ", [f.id for f in deps.knowledge.search("How do partners get sandbox access?")], "match a sandbox question")

preference  meeting_length_minutes   '30'                         (seeded)
preference  reply_length             'medium'                     (seeded)
preference  sign_off                 'Best regards, Erland'       (seeded)
preference  working_hours            '09:00-17:00'                (seeded)
contact     sofia.lind@example.com           manager, Head of Product  VIP
contact     mark.jensen@nordwind.example     partner, Nordwind Payments


org facts   ['F02', 'F03'] match a sandbox question


## 2. Process the inbox

The inbox has 15 emails in 10 threads. Only the latest unread message of each thread is processed.
Each email runs through the LangGraph orchestrator: triage, then a deterministic router, then a worker.
Every outgoing action stops at `interrupt()` for a human decision. Here the decisions are scripted:

| Email | Decision |
|---|---|
| e04 (partner asks about sandbox access) | **edit** the draft: shorter, sign-off "Best, Erland" |
| e03 (manager asks for KPI numbers) | **reject**: "I'll answer Sofia in person" |
| e07 (colleague asks for a meeting) | approve the proposed slot |
| e09 (partner will come back about a contract) | approve the reminder task |
| e15 (a second partner asks about sandbox access) | approve, after looking at the draft |

In [3]:
SHORT_E04 = "Hi Mark,\n\nSandbox credentials come via the partner portal; I've asked for access for your team.\n\nBest, Erland"
views = {}  # the approval views, kept to look at below


def decide(view):
    email_id = view["email"]["id"]
    views[email_id] = view
    if email_id == "e04":
        payload = {**view["proposal"]["payload"], "body": SHORT_E04}
        return HumanDecision(decision="edit", edited_payload=payload, comment="Shorter, and my usual sign-off")
    if email_id == "e03":
        return HumanDecision(decision="reject", comment="I'll answer Sofia in person")
    return HumanDecision(decision="approve")


states = {}
for email in emails_to_process(deps):
    states[email.id] = process_email(app, deps, email.id, decide)


def outcome(state):
    if state.get("route") == "human_review":
        return "REVIEW: " + state["review_reason"]
    return state.get("outcome") or "labelled and archived"


print(f"{'id':4} {'intent':16} {'priority':8} {'conf':5} {'labels':45} outcome")
for email_id, s in states.items():
    t = s["triage"]
    labels = ", ".join(deps.mail.get_email(email_id).labels)
    print(f"{email_id:4} {t.intent.value:16} {t.priority.value:8} {t.confidence:<5.2f} {labels[:45]:45} {outcome(s)}")

id   intent           priority conf  labels                                        outcome
e04  reply_needed     medium   0.95  Partners                                      Done: email to mark.jensen@nordwind.example
e09  follow_up        medium   0.80  Legal                                         Done: task 'Follow up: Data processing agreement for signature'
e12  spam             low      0.95  Promotions, Spam                              labelled and archived
e10  newsletter       low      1.00  Newsletters                                   labelled and archived
e13  reply_needed     medium   0.50  Finance, Possible fraud: verify, Lookalike do REVIEW: Possible fraud: unknown sender asks to change payment details. Verify by phone (F05).
e03  reply_needed     high     0.80  Internal, Reporting                           Rejected: I'll answer Sofia in person
e07  meeting_request  medium   0.65  Internal, Meetings                            Done: event 'Sync on the recommendations roa

## 3. Reply flow: summarise, draft, edit, approve, send

For e04 the summariser condensed the thread, and the drafter used tools (contact lookup, knowledge search,
preferences) before writing. This is the approval view the human saw: triage, summary, the proposal,
and the organisational facts the draft relied on.

In [4]:
print(render_view(views["e04"]))

Email e04 from mark.jensen@nordwind.example: API sandbox access for the Nordwind integration
Triage: reply_needed, priority medium, confidence 0.95, tags Partners
Why: Keyword match for reply_needed: ['?', 'could you']
Summary: Thread 'API sandbox access for the Nordwind integration' with 1 message(s) from mark.jensen@nordwind.example.
  Open question: Could you send us sandbox credentials and a link to the API docs?
  Open question: Also, what are your support hours if our developers get stuck?
Proposed action: send_email (Reply drafted for an email that needs an answer.)
  To: mark.jensen@nordwind.example
  Subject: Re: API sandbox access for the Nordwind integration

Hi Mark,

Thanks for your email.
Partner API sandbox credentials are requested through the partner portal at https://partners.example.com and are issued within two business days.
The partner API documentation is at https://developers.example.com/partner-api.
Partner support hours are weekdays 09:00 to 17:00 CET. Product

In [5]:
sent = json.loads((RUNTIME / "sent.json").read_text(encoding="utf-8"))
for record in sent:
    print(f"SENT {record['action_id']} to {', '.join(record['to'])}: {record['subject']}")
    print("   ", record["body"].replace("\n", "\n    "))

SENT act-e807c7572aa2bce6 to mark.jensen@nordwind.example: Re: API sandbox access for the Nordwind integration
    Hi Mark,
    
    Sandbox credentials come via the partner portal; I've asked for access for your team.
    
    Best, Erland
SENT act-dead59f8856d2655 to anna.holm@fjordanalytics.example: Re: Sandbox access for Fjord Analytics
    Hi Anna,
    
    Thanks.
    Partner API sandbox credentials are requested through the partner portal at https://partners.example.com and are issued within two business days.
    
    Best, Erland


## 4. What the copilot learned

Long-term memory is written only from human decisions, never from email content (DECISIONS.md, D30).
The e04 edit taught two preferences; the e03 rejection is stored as feedback.

In [6]:
for item in states["e04"].get("learned", []):
    print(f"learned from the e04 edit: {item.key} = {item.value!r}")
print()
for p in deps.memory.preference_records():
    print(f"preference  {p.key:24} {p.value!r:28} ({p.source})")
print()
for f in deps.memory.all_feedback():
    print(f"feedback    {f.email_id}: {f.decision}{' - ' + f.comment if f.comment else ''}")

learned from the e04 edit: sign_off = 'Best, Erland'
learned from the e04 edit: reply_length = 'short'

preference  meeting_length_minutes   '30'                         (seeded)
preference  reply_length             'short'                      (learned)
preference  sign_off                 'Best, Erland'               (learned)
preference  working_hours            '09:00-17:00'                (seeded)

feedback    e04: edit - Shorter, and my usual sign-off
feedback    e09: approve
feedback    e03: reject - I'll answer Sofia in person
feedback    e07: approve
feedback    e15: approve


## 5. Memory at work: a similar email gets the new style

e15 arrived after the e04 edit. Its draft was written with the learned preferences (short, "Best, Erland").

In [7]:
print(views["e15"]["proposal"]["payload"]["body"])

Hi Anna,

Thanks.
Partner API sandbox credentials are requested through the partner portal at https://partners.example.com and are issued within two business days.

Best, Erland


## 6. Scheduling

The scheduler checked the calendar within working hours from preferences (09:00 to 17:00, Europe/Stockholm),
proposed a free slot, and after approval the event and the reminder task were written to the mock calendar.

In [8]:
calendar = json.loads((RUNTIME / "calendar.json").read_text(encoding="utf-8"))
for item in [*calendar["events"], *calendar["tasks"]]:
    if item.get("action_id"):
        print(f"{item['action_id']}  {item['title']}  {item.get('start') or item.get('due')}")

act-ca498c89753b9a89  Sync on the recommendations roadmap  2026-10-27T15:00:00+01:00
act-1a5e456d752114f7  Follow up: Data processing agreement for signature  2026-10-27T09:00:00+01:00


## 7. Newsletters and spam: labelled and archived, never deleted

In [9]:
for email_id in ["e10", "e12", "e11"]:
    e = deps.mail.get_email(email_id)
    print(f"{email_id} archived={e.archived} labels={e.labels}  ({e.subject})")

e10 archived=True labels=['Newsletters']  (AI Engineering Weekly, issue #142)
e12 archived=True labels=['Promotions', 'Spam']  (Congratulations! You have been selected)
e11 archived=True labels=['Internal', 'IT']  (Office network maintenance on Saturday)


## 8. The human review queue

Low-confidence results and fraud warnings are not acted on. They get a "Needs review" label and a reason.

In [10]:
for email_id, s in states.items():
    if s.get("route") == "human_review":
        print(f"{email_id}: {s['review_reason']}")

e13: Possible fraud: unknown sender asks to change payment details. Verify by phone (F05).
e14: Low confidence (0.50) for meeting_request.


## 9. Exactly once

Running an email again does nothing: the run is finished in the checkpointer, and every action has a
deterministic `action_id` that the mock adapters check before acting.

In [11]:
before = len(json.loads((RUNTIME / "sent.json").read_text(encoding="utf-8")))
process_email(app, deps, "e04", decide)
after = len(json.loads((RUNTIME / "sent.json").read_text(encoding="utf-8")))
print(f"sent before: {before}, after running e04 again: {after}")

sent before: 2, after running e04 again: 2


## 10. A provider outage

First without a fallback: the drafter's model call fails, the agent retries once, and the email goes to human
review instead of guessing. The error is logged with its trace id in `errors.jsonl`.
Then with a fallback provider: the same failure is absorbed and the run continues. (Simulated here with a
broken fake provider; with real keys `build_deps` sets this up between OpenAI and Anthropic.)

In [12]:
from assistant.llm import FakeLLM, LLMError
from assistant.llm.fallback import FallbackLLM
from assistant.llm.instrumented import InstrumentedLLM


class Outage(FakeLLM):
    provider = "broken"

    def next_step(self, *args, **kwargs):
        raise LLMError("503 Service Unavailable")


healthy_llm = deps.llm
fresh = build_graph()  # a new checkpointer, so e03 can be processed again from the start

deps.llm = InstrumentedLLM(Outage(), deps.obs)
print("without fallback:", outcome(process_email(fresh, deps, "e03")))

deps.llm = FallbackLLM(InstrumentedLLM(Outage(), deps.obs), healthy_llm, {}, deps.obs)
state = process_email(build_graph(), deps, "e03")
print("with fallback:   ", "waiting for approval" if pending_view(state) else outcome(state))
deps.llm = healthy_llm

without fallback: REVIEW: Drafting failed after a retry; see errors.jsonl.


with fallback:    waiting for approval


## 11. Observability

Metrics for this run, then the trace of the meeting request as a waterfall
(the same as `python scripts/show_trace.py e07`).

In [13]:
deps.obs.close()  # flush spans and write metrics.json
print(deps.obs.metrics.summary())

Metrics
  decisions_total{decision=approve}                          3
  decisions_total{decision=edit}                             1
  decisions_total{decision=reject}                           1
  email_runs_total{route=archive,status=done}                3
  email_runs_total{route=human_review,status=done}           3
  email_runs_total{route=reply,status=done}                  3
  email_runs_total{route=reply,status=waiting_for_human}     1
  email_runs_total{route=schedule,status=done}               2
  emails_by_intent_total{intent=follow_up}                   1
  emails_by_intent_total{intent=fyi}                         1
  emails_by_intent_total{intent=meeting_request}             2
  emails_by_intent_total{intent=newsletter}                  1
  emails_by_intent_total{intent=reply_needed}                6
  emails_by_intent_total{intent=spam}                        1
  errors_total{node=draft}                                   1
  llm_calls_total{model=fake-draft,provider=bro

In [14]:
from assistant.observability.waterfall import find_trace, load_spans, render

spans = load_spans(LOGS / "traces.jsonl")
print(render(spans, find_trace(spans, "e07")))
print()
print(render(spans, find_trace(spans, "e03")))  # the fallback run: a failed call, then a good one

Trace 096eefd8b310f864570b769b08a11d67  email e07  run ee7455e37a55  22.4 ms
email_run                                |##############################|    22.4 ms
  node fetch                             |  #                           |     0.1 ms
  node triage                            |   #                          |     0.9 ms
    chat fake-triage                     |   #                          |     0.2 ms  233 in / 7 out
  node schedule                          |     ###########              |     8.2 ms
    chat fake-draft                      |     #                        |     0.1 ms
    execute_tool get_preferences         |      #                       |     0.4 ms
    chat fake-draft                      |      #                       |     0.1 ms
    execute_tool check_availability      |      #######                 |     5.2 ms
    chat fake-draft                      |              ##              |     1.2 ms  298 in / 14 out
  node propose_action                   

In [15]:
for name in ["app.jsonl", "errors.jsonl", "traces.jsonl", "metrics.json"]:
    path = LOGS / name
    print(f"{path.relative_to(ROOT)}: {sum(1 for _ in path.open(encoding='utf-8'))} lines")
print()
print(next(line for line in (LOGS / "errors.jsonl").open(encoding="utf-8")))

logs/demo/app.jsonl: 149 lines
logs/demo/errors.jsonl: 1 lines
logs/demo/traces.jsonl: 139 lines
logs/demo/metrics.json: 125 lines

{"node": "draft", "error": "drafter: no valid DraftReply after one retry: 503 Service Unavailable", "event": "node_error", "run_id": "4bd31e538afd", "email_id": "e03", "level": "error", "timestamp": "2026-10-22T15:55:07.427528Z", "trace_id": "a8a8fd9f8e5b557c16830c3330087ca3", "span_id": "02757f842e3b080a"}

